# Part 2: Automated Data Quality Audit & Point-in-Time Panel Construction
Audits cross-quarter schema drift, structural missingness by regulatory era, balance-sheet invariants, YTD flow de-cumulation, and bank exit classification (FDIC failure vs. non-failure mergers).

In [1]:
from pathlib import Path
import polars as pl

dq_text = Path('../outputs/dq_report/data_quality_report.md').read_text()
print(dq_text[:2500] + '\n...[see outputs/dq_report/data_quality_report.md for full report]...')

# Automated Data Quality Audit (Part 2)

Generated by `src/bfews/quality.py`; every number below is computed from the raw files.

## 1. Completeness
- Quarterly files: **170** (1984Q1-2026Q2); rows: **1,682,616**
- Quarters with `rows != api_total` or flagged incomplete: **0**

## 2. Schema drift across quarterly files
14 columns change physical type between files (int64 / double / null), so a naive `pl.scan_parquet('*.parquet')` fails with `SchemaError`.

| column | types |
|---|---|
| CHBAL | double / int64 |
| DEP | double / int64 |
| DEPDOM | double / int64 |
| LNCI | double / int64 |
| LNCON | double / int64 |
| LNLSGR | double / int64 |
| LNLSNET | double / int64 |
| LNRE | double / int64 |
| NAASSET | double / int64 |
| NCLNLS | double / int64 |
| P3ASSET | int64 / null |
| P9ASSET | double / int64 |
| RBCRWAJ | double / int64 |
| SC | double / int64 |

## 3. Structural missingness by era (% null)
| era | rows | P3ASSET null % | P9ASSET null % | NAASSET null % | ELNATR null % | 

In [2]:
exit_df = pl.read_csv('../outputs/tables/bank_exit_summary.csv')
print('=== Bank Exit Taxonomy ===')
print(exit_df)
panel_schema = pl.read_parquet_schema('../data/silver/panel.parquet')
print(f'Silver panel columns ({len(panel_schema)}):', list(panel_schema.keys())[:15], '...')

=== Bank Exit Taxonomy ===
shape: (3, 4)
┌─────────────────┬───────┬───────────────┬───────────────────────────┐
│ exit_status     ┆ banks ┆ bank_quarters ┆ banks_with_failure_record │
│ ---             ┆ ---   ┆ ---           ┆ ---                       │
│ str             ┆ i64   ┆ i64           ┆ i64                       │
╞═════════════════╪═══════╪═══════════════╪═══════════════════════════╡
│ MERGED_OR_OTHER ┆ 16892 ┆ 923166        ┆ 375                       │
│ ACTIVE          ┆ 4313  ┆ 671777        ┆ 7                         │
│ FAILED          ┆ 2859  ┆ 87673         ┆ 2859                      │
└─────────────────┴───────┴───────────────┴───────────────────────────┘
Silver panel columns (44): ['CERT', 'REPDTE', 'STNAME', 'ASSET', 'DEP', 'DEPDOM', 'EQ', 'BRO', 'CHBAL', 'SC', 'LNLSGR', 'LNLSNET', 'LNRE', 'LNCI', 'LNCON'] ...
